# Retrieval: 문서를 검색 가능한 지식으로 바꾸고 RAG로 연결하기

Retrieval은 사용자의 질문과 관련된 정보를 외부 문서에서 찾아 `Document` 목록으로 반환하는 과정이다.

```
Retrieval: 검색 과정
Retriever: 검색 객체
```


LLM(Large Language Model, 대규모 언어 모델)이 학습하지 않은 사내 문서나 최신 자료를 답변에 사용하려면, 질문을 바로 모델에 보내기 전에 필요한 근거를 검색해야 한다.

- 웹페이지 또는 PDF를 가져와 LangChain의 `Document`로 표준화
- 긴 문서를 chunk로 나누고, OpenAI와 Hugging Face 임베딩으로 숫자 벡터를 만든 뒤, Vector Store와 Retriever를 거쳐 2-step RAG 답변까지 연결한다.

## Retrieval와 환각의 관계

환각(Hallucination)은 LLM이 확인된 근거 없이 그럴듯한 내용을 생성하는 현상이다. 학습 데이터에 정보가 없거나 질문의 맥락이 부족해도 모델은 다음 토큰을 계속 예측하므로, 존재하지 않는 출처나 잘못된 사실을 만들 수 있다.

Retrieval-Augmented Generation(RAG, 검색 증강 생성)은 답변 전에 외부 문서를 검색하고, 검색된 내용을 모델의 문맥으로 전달한다. RAG가 환각을 완전히 제거하지는 않지만 어떤 문서를 근거로 사용했는지 확인할 수 있고, 문맥에 없는 내용은 답하지 않도록 제어할 수 있다.

- Retrieval은 관련 근거를 찾는 단계이다.
- Generation은 검색된 근거를 읽고 답변을 만드는 단계이다.
- 검색 결과가 부정확하면 생성 결과도 부정확해질 수 있으므로 최종 답변보다 먼저 검색 문서를 확인해야 한다.

[LangChain Retrieval 공식 문서](https://docs.langchain.com/oss/python/langchain/retrieval)에서 Retrieval 구성 요소와 RAG 구조를 확인할 수 있다.

## Retrieval 파이프라인을 먼저 읽기

Retrieval은 서로 다른 역할의 구성 요소를 순서대로 연결한다.
각 단계의 출력이 다음 단계의 입력이 되므로 하나의 객체 이름만 외우기보다 데이터 형태가 어떻게 바뀌는지 추적해야 한다.

`외부 데이터 → list[Document] → list[chunk Document] → embedding vector → Vector Store → Retriever → list[Document] → LLM 답변`

- **Document Loader**: 웹, PDF, 파일처럼 서로 다른 원본을 `Document`로 표준화한다.
- **Text Splitter**: 긴 Document를 검색 가능한 작은 Document로 나눈다.
- **Embedding Model**: 문서와 질문을 의미를 표현하는 숫자 벡터로 바꾼다.
- **Vector Store**: 벡터와 원문·metadata를 보관하고 유사도 검색을 수행한다.
- **Retriever**: 문자열 질문을 받아 관련된 `list[Document]`를 반환한다.
- **2-step RAG**: 검색을 먼저 실행한 뒤 그 결과를 LLM 문맥으로 전달한다.

구성 요소가 분리되어 있으므로 웹 대신 PDF를 사용하거나 InMemoryVectorStore 대신 Chroma를 사용해도 앞뒤 데이터 계약을 유지할 수 있다.

## 패키지 설치

Python 패키지는 특정 기능을 재사용할 수 있도록 묶어 PyPI에서 배포하는 코드 단위이다. LangChain은 모든 기능을 한 패키지에 넣지 않고, 공통 인터페이스와 외부 기술별 연동 패키지를 나누어 제공한다.

- `langchain-text-splitters`: 긴 텍스트를 검색 단위인 chunk로 나누는 알고리즘을 제공한다.
- `langchain-huggingface`: Hugging Face 모델을 LangChain의 Embeddings 인터페이스로 연결한다.
- `langchain-chroma`: LangChain의 `Document`와 임베딩을 Chroma에 저장하고 검색하게 한다.
- `beautifulsoup4`: HTML을 태그 구조로 해석하는 파서이다. 설치 이름은 `beautifulsoup4`이고 import 이름은 `bs4`이다.
- `pypdf`: PDF의 페이지 구조를 읽고 텍스트를 추출하는 라이브러리이다.
- `gdown`: 공개 Google Drive 파일 ID를 이용해 파일을 내려받는 도구이다.
- `sentence-transformers`: 문장이나 문단을 의미 벡터로 변환하는 로컬 임베딩 모델 실행 프레임워크이다.

`-U`는 이미 설치된 패키지가 있으면 현재 PyPI의 최신 호환 버전으로 갱신한다. 설치 후 import 오류가 계속되면 커널을 한 번 재시작한다.

In [23]:
%pip install -U langchain-text-splitters langchain_huggingface langchain_chroma beautifulsoup4 pypdf gdown sentence-transformers ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Retrieval 구현에 사용하는 클래스

Retrieval 코드는 원문을 읽는 객체, 텍스트를 벡터로 바꾸는 객체, 벡터를 검색하는 객체를 연결한다.

`langchain_core`는 이 객체들이 주고받을 `Document`와 공통 인터페이스를 정의하고, 공급자별 패키지는 실제 서비스나 라이브러리에 연결되는 구현체를 제공한다.

- 데이터 수집·변환: `BeautifulSoup`, `PdfReader`, `RecursiveCharacterTextSplitter`가 원문을 읽고 검색 단위로 바꾼다.
- 임베딩: `OpenAIEmbeddings`와 `HuggingFaceEmbeddings`가 텍스트를 `list[float]` 벡터로 바꾼다.
- 저장·검색: `InMemoryVectorStore`와 `Chroma`가 벡터와 `Document`를 저장하고 관련 문서를 찾는다.
- 답변 생성: `ChatPromptTemplate`, `ChatOpenAI`, `StrOutputParser`가 검색 결과를 모델 입력으로 만들고 최종 문자열을 반환한다.

이후 셀에서는 각 객체를 한꺼번에 사용하지 않고 `수집 → 분할 → 임베딩 → 저장 → 검색 → 생성` 순서로 하나씩 연결한다.

In [24]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

CHAT_MODEL_NAME = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')

print('OPENAI_API_KEY configured:', bool(os.getenv('OPENAI_API_KEY')))
print('chat model:', CHAT_MODEL_NAME)

OPENAI_API_KEY configured: True
chat model: gpt-5.6-luna


## Document: 본문과 출처를 함께 보관하는 표준 객체

`Document`는 Retrieval 파이프라인에서 문서 한 개를 표현하는 LangChain의 표준 자료형이다. 파일 종류가 달라도 Document로 바꾸면 이후 Text Splitter, Vector Store와 Retriever가 같은 방식으로 처리할 수 있다.

- `page_content`: 검색과 임베딩에 사용할 실제 텍스트 문자열이다.
- `metadata`: 출처, 페이지, 작성자처럼 검색 결과를 설명하고 필터링할 때 사용할 딕셔너리이다.
- `id`: Document를 수정하거나 삭제할 때 사용할 수 있는 선택 식별자(Identifier)이다.

먼저 작은 Document를 직접 만들어 두 속성의 경계를 확인한다. 실제 웹과 PDF도 같은 구조로 변환한다.

In [25]:
page_content='톰 소여는 미시시피강 주변에서 모험을 겪는 소년이다.'
metadata={
    'source': 'The Adventures of Tom Sawyer',
    'page': 1,
    'author': 'Mark Twain',
}

sample_document = Document(
    page_content=page_content, # 본문(문자열)
    metadata=metadata, # 추가 정보, 본문과 섞이지 않는 정보
)

print("type:", type(sample_document))
print("page_content:", sample_document.page_content)
print("metadata:", sample_document.metadata)

type: <class 'langchain_core.documents.base.Document'>
page_content: 톰 소여는 미시시피강 주변에서 모험을 겪는 소년이다.
metadata: {'source': 'The Adventures of Tom Sawyer', 'page': 1, 'author': 'Mark Twain'}


## 공개 웹페이지를 Document로 변환하기

Document Loader는 외부 데이터에서 내용을 읽고 `Document`를 반환하는 역할을 뜻한다. 반드시 특정 Loader 클래스를 사용해야 하는 것은 아니며, 원문을 읽어 `page_content`와 `metadata`로 변환하면 같은 역할을 수행할 수 있다.

`requests`는 URL로 HTTP(HyperText Transfer Protocol) 요청을 보내고 상태 코드와 HTML 문자열이 담긴 `Response`를 반환한다. `BeautifulSoup`은 이 HTML 문자열을 태그 트리로 해석해 필요한 텍스트를 선택하거나 불필요한 태그를 제거한다. BeautifulSoup은 브라우저가 아니므로 JavaScript를 실행하지 않는다.

이어지는 코드는 IANA(Internet Assigned Numbers Authority) Reserved Domains 페이지에 다음 변환을 적용한다.
- IANA: 인터넷에서 사용하는 도메인 이름, IP 주소, 프로토콜 번호 같은 고유 식별자를 관리하는 기관

`URL → HTTP Response → HTML 태그 구조 → 정리된 문자열 → Document`

`script`와 `style`을 제거한 본문은 `page_content`에, URL과 제목은 `metadata`에 저장한다. JavaScript로 본문을 만드는 사이트라면 브라우저 기반 수집 도구가 별도로 필요하다.

In [26]:
import requests
from bs4 import BeautifulSoup

source_url = 'https://www.iana.org/domains/reserved'

# 지정된 주소로 요청을 보내서 해당 웹 페이지 응답 받기
# 단, 10초 내로 응답이 오지 않으면 에러 발생
web_response = requests.get(
    url=source_url,
    timeout=10
)

web_response.raise_for_status() # 4xx, 5xx 응답 상태 코드를 받으면 에러 발생

# BeautifulSoup() 객체를 이용하여 HTML 문자열을 태그 트리 구조로 변환
soup = BeautifulSoup(web_response.text, 'html.parser')

# html 태그 중 script, style 태그를 찾아서 제외
for element in soup(['script', 'style']):
    element.decompose()

# 연속 공백과 줄바꿈을 하나의 공백으로 처리
page_text = ' '.join(soup.get_text(' ', strip=True).split())

# 문서 제목 얻어오기
page_title = soup.title.get_text(strip=True) if soup.title else 'IANA Reserved Domains'

web_document = Document(
    page_content=page_text,
    metadata={
        'source': source_url,
        'title': page_title,
        'source_type': 'web',
    }
)

# List로 변경
web_documents: list[Document] = [web_document]

print("Document Count:", len(web_documents))
print("page_content preview:", web_documents[0].page_content[:400])
print("metadata:", web_documents[0].metadata)

Document Count: 1
page_content preview: IANA-managed Reserved Domains Domains Protocols Numbers About IANA-managed Reserved Domains Certain domains are set aside, and nominally registered to “IANA”, for specific policy or technical purposes. Example domains As described in RFC 2606 and RFC 6761 , a number of domains such as example.com and example.org are maintained for documentation purposes. These domains may be used as illustrative e
metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web'}


## 실제 PDF 파일 내려받기

PDF(Portable Document Format)를 처리하려면 먼저 Python이 읽을 수 있는 로컬 파일 경로가 필요하다.

`gdown`은 Google Drive 공유 페이지를 직접 조작하는 대신 공개 파일 ID를 사용해 파일을 내려받는 Python 도구이다.

- `id`: Google Drive가 파일을 구분하는 식별자이다.
- `output`: 다운로드한 파일을 저장할 로컬 경로이다.
- `quiet=False`: 다운로드 진행 상태를 화면에 표시한다.


In [27]:
import gdown

pdf_directory = Path('data')
pdf_directory.mkdir(parents=True, exist_ok=True)
pdf_path = pdf_directory / 'The_Adventures_of_Tom_Sawyer.pdf'

downloaded_path = gdown.download(
    id='1gtuIlwvNeI9rtHmBnU8RGUrQ65DpLBnh',
    output=str(pdf_path),
    quiet=False,
)

if downloaded_path is None:
    raise RuntimeError('PDF 다운로드 실패')

print('downloaded path:', downloaded_path)
print('file size (bytes):', pdf_path.stat().st_size)

Downloading...
From: https://drive.google.com/uc?id=1gtuIlwvNeI9rtHmBnU8RGUrQ65DpLBnh
To: C:\SKN_AI\09_llm\05_langchain\02_langchain_component\data\The_Adventures_of_Tom_Sawyer.pdf
100%|██████████| 2.68M/2.68M [00:00<00:00, 9.68MB/s]

downloaded path: data\The_Adventures_of_Tom_Sawyer.pdf
file size (bytes): 2684971


## PDF 페이지를 Document로 변환하기

`pypdf`는 PDF의 페이지, 문서 정보와 텍스트 계층을 읽는 Python 라이브러리이다. `PdfReader`는 PDF 파일을 열어 `pages`에 페이지 객체를 제공하고, 각 페이지의 `extract_text()`는 내장된 텍스트를 문자열로 반환한다.

페이지마다 `Document`를 만들면 검색된 문장이 어느 페이지에서 왔는지 `metadata`로 추적할 수 있다. 처리 흐름은 다음과 같다.

`PDF 파일 → PdfReader → 페이지 객체 → 텍스트 문자열 → 페이지별 Document`

`extract_text()`는 문자 정보가 들어 있는 PDF에서만 안정적으로 동작한다. 스캔 이미지로만 구성된 PDF에는 OCR(Optical Character Recognition, 광학 문자 인식)이나 레이아웃 인식 기능이 있는 별도 도구가 필요하다.

In [9]:
from pypdf import PdfReader

pdf_reader = PdfReader(pdf_path) # pdf_path == pdf 경로
total_pages = len(pdf_reader.pages)

# Document로 변환된 PDF 페이지를 모아둘 List
pdf_documents: list[Document] = []

for page_number, pdf_page in enumerate(pdf_reader.pages, start=1):
    # extract_text()가 None을 반환하는 이미지 전용 페이지는 빈 문자열로 바꾼다.
    page_text = pdf_page.extract_text() or ''
    if not page_text.strip():
        continue

    # page_number는 사람이 읽는 1부터 시작하며, source는 검색 결과의 파일 근거로 사용한다.
    pdf_documents.append(
        Document(
            page_content=page_text,
            metadata={
                'source': str(pdf_path),
                'page': page_number,
                'total_pages': total_pages,
                'source_type': 'pdf',
            },
        )
    )


# Web과 PDF의 Document를 하나의 List에 합치기
source_documents: list[Document] = [*web_documents, *pdf_documents]

print('PDF total pages:', total_pages)
print('PDF Document count:', len(pdf_documents))
print('page 11 metadata:', pdf_documents[10].metadata)
print('page 11 preview:', pdf_documents[10].page_content[:500])
print('combined source count:', len(source_documents))


PDF total pages: 35
PDF Document count: 34
page 11 metadata: {'source': 'data\\The_Adventures_of_Tom_Sawyer.pdf', 'page': 12, 'total_pages': 35, 'source_type': 'pdf'}
page 11 preview:  
 
 
Peter had some medicine. He didn’t like it! 
combined source count: 35


## 긴 Document를 검색용 chunk로 나누기

Chunk는 긴 문서에서 잘라낸 작은 검색 단위이다. 페이지 전체를 하나의 벡터로 만들면 질문과 무관한 문장이 섞이고, 지나치게 작게 나누면 답변에 필요한 앞뒤 문맥이 끊길 수 있다.

`RecursiveCharacterTextSplitter`는 문단, 줄바꿈, 공백처럼 큰 의미 경계를 먼저 시도한 뒤 크기 제한을 넘을 때 더 작은 경계로 재귀적으로 나눈다. 기본 길이 기준은 토큰 수가 아니라 Python `len()`으로 센 문자 수이다.

예를 들어 1,000자 본문에 `chunk_size=800`, `chunk_overlap=120`을 적용하면 하나의 chunk에 최대 800자를 담고 다음 chunk 앞부분에 이전 경계의 최대 120자를 다시 포함한다. 겹침은 문맥 손실을 줄이지만 중복 저장량과 임베딩 비용을 늘린다.

분할된 결과도 `Document`이므로 원본의 URL·페이지 metadata를 유지하며, `add_start_index=True`는 원문에서 각 chunk가 시작한 문자 위치를 추가한다.

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,

    # 잘라낸 chunk가 원문에서 시작한 문자 위치를 메타데이더로 추가
    add_start_index=True,
)

# 웹 + PDF가 합쳐진 Document List를 청크 단위로 쪼개기
chunks: list[Document] = text_splitter.split_documents(source_documents)

first_chunk = chunks[0]

print('source Document count:', len(source_documents)) # 페이지 수
print('chunk count:', len(chunks)) #전체 chunk 개수를
print('first chunk characters:', len(first_chunk.page_content)) # 첫 번째 chunk의 실제 문자 수
print('first chunk metadata:', first_chunk.metadata) # 첫 번째 chunk의 출처 정보를 출력
print('first chunk preview:', first_chunk.page_content[:400]) # 첫 번째 chunk의 본문 중 앞 400자만 출력


source Document count: 35
chunk count: 61
first chunk characters: 799
first chunk metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web', 'start_index': 0}
first chunk preview: IANA-managed Reserved Domains Domains Protocols Numbers About IANA-managed Reserved Domains Certain domains are set aside, and nominally registered to “IANA”, for specific policy or technical purposes. Example domains As described in RFC 2606 and RFC 6761 , a number of domains such as example.com and example.org are maintained for documentation purposes. These domains may be used as illustrative e


## Embedding: 텍스트를 의미 벡터로 바꾸기

임베딩(Embedding)은 텍스트의 의미적 특징을 여러 실수로 표현한 고정 길이 벡터이다. Dense Embedding은 문장 하나를 대부분의 값이 채워진 하나의 벡터로 표현하며, 각 좌표를 사람이 특정 단어의 점수처럼 직접 해석하지는 않는다.

예를 들어 “domains reserved for documentation”과 “domain names used in examples”는 사용한 단어가 완전히 같지 않아도 의미가 비슷하므로 임베딩 공간에서 가까운 벡터가 될 수 있다. 문서와 질문을 같은 모델로 변환해야 같은 좌표계에서 코사인 유사도, 유클리드 거리 또는 내적을 계산할 수 있다.

LangChain Embeddings 인터페이스는 공급자가 달라도 다음 두 메서드를 공통으로 제공한다.

- `embed_query(text)`: 질문 문자열 하나를 `list[float]` 하나로 변환한다.
- `embed_documents(texts)`: 여러 문서 문자열을 같은 순서의 `list[list[float]]`로 변환한다.

### OpenAI Embedding

- OpenAI 서버에서 모델을 실행하므로 로컬 컴퓨터의 연산 부담이 적다.
- API 키와 인터넷 연결이 필요하며 입력 토큰에 따라 비용이 발생한다.
- 임베딩할 텍스트가 API 요청을 통해 외부 서버로 전달된다.
- 모델을 직접 다운로드하거나 실행 환경을 관리하지 않고 빠르게 적용할 때 적합하다.

### Hugging Face 로컬 임베딩

- 모델을 내려받은 뒤 현재 컴퓨터의 CPU 또는 GPU에서 실행한다.
- 모델 호출별 API 비용은 없지만 최초 다운로드와 메모리·연산 자원이 필요하다.
- 문서가 외부 임베딩 API로 전송되지 않아 로컬 데이터 처리가 필요한 경우에 적합하다.
- Hugging Face는 모델 저장소이며, 실제 품질·언어·벡터 차원은 선택한 모델마다 다르다.

### 공통 주의사항

문서와 질문은 반드시 같은 임베딩 모델로 변환해야 한다. OpenAI 벡터와 Hugging Face 벡터는 차원과 좌표 공간이 다를 수 있으므로 서로 직접 비교할 수 없다. [Embedding 공식 문서](https://docs.langchain.com/oss/python/integrations/embeddings)에서 공통 인터페이스와 선택 기준을 확인할 수 있다.

### OpenAI 임베딩으로 질문 하나 변환하기

`OpenAIEmbeddings`는 텍스트를 OpenAI Embedding API에 보내 검색과 유사도 계산에 사용할 숫자 벡터를 반환하는 연동 클래스이다. 답변 문장을 생성하는 `ChatOpenAI`와 공급자는 같지만 역할과 출력이 다르다.

- `OpenAIEmbeddings`: 문자열을 입력받아 `list[float]`를 반환한다.
- `ChatOpenAI`: 메시지를 입력받아 모델의 답변이 담긴 `AIMessage`를 반환한다.

다음 셀은 질문 하나를 `text-embedding-3-small`로 변환하고 자료형, 벡터 차원과 앞부분 값을 확인한다. 실제 API 키·네트워크·입력 토큰 비용이 필요하다.

In [12]:
embedding_example_text = 'Which domains are reserved for documentation examples?'

# 임베딩 모델 선택
openai_embedding_model = "text-embedding-3-small"

# 임베딩 모델 생성
openai_embeddings = OpenAIEmbeddings(model=openai_embedding_model)

# embed_query() : 문자열 하나를 받아서 query용 vector를 얻는다
openai_query_vector : list[float] = (
    openai_embeddings.embed_query(embedding_example_text)
)
print(len(openai_query_vector)) # 벡터 차원 수 == 1536
print(openai_query_vector[:5])

1536
[0.0121612548828125, 0.00860595703125, 0.05419921875, -0.012603759765625, 0.019866943359375]


### Hugging Face 모델로 로컬 임베딩 만들기

Hugging Face는 모델 파일과 모델 카드를 공유하는 생태계이고, Sentence Transformers는 문장·문단 전체를 의미 벡터로 변환하도록 학습된 모델을 실행하는 프레임워크이다. `HuggingFaceEmbeddings`는 Sentence Transformers 모델을 LangChain의 `embed_query()`와 `embed_documents()` 인터페이스로 감싸는 연동 클래스이다.

`sentence-transformers/all-MiniLM-L6-v2`는 Hugging Face 저장소에서 모델을 찾는 ID이다. 비교적 가볍고 빠른 영어 임베딩 모델이며 384차원 벡터를 반환한다. Hugging Face에 등록된 모델 중 문장 임베딩 용도로 학습된 모델을 선택해야 한다.

첫 실행에는 모델 가중치 다운로드가 필요하지만 API 키와 건당 호출 비용은 없다. 이 실습은 CPU(Central Processing Unit, 중앙 처리 장치)에서 실행한다. 한국어 문서를 검색하려면 한국어 또는 다국어 학습 여부와 query·document prefix 사용법을 모델 카드에서 확인해야 한다. [Sentence Transformers 공식 통합 문서](https://docs.langchain.com/oss/python/integrations/embeddings/sentence_transformers)에서 로컬 실행 방식을 확인할 수 있다.

In [29]:
local_embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},

    # 벡터 길이를 1로 맞춰 코사인 기반 비교를 단순화 시킨다
    encode_kwargs={'normalize_embeddings': True},
)

local_query_vector : list[float] = (
    local_embeddings.embed_query(embedding_example_text)
)


print(len(local_query_vector))
print(local_query_vector[:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384
[-0.026091985404491425, -0.06857788562774658, -0.019899435341358185, -0.0650050938129425, 0.011259421706199646]


### 여러 chunk를 한 번에 임베딩하기

검색용 인덱스를 만들 때는 질문 하나가 아니라 모든 chunk를 벡터로 변환해야 한다. `embed_documents()`는 문자열 목록을 입력받으며 입력 문자열 하나마다 벡터 하나를 같은 순서로 반환한다.

전체 chunk를 실제 로컬 모델로 임베딩한다. 출력 벡터의 개수가 chunk 개수와 같은지 확인하면 문서 누락 여부를 점검할 수 있다.

In [15]:
import pandas as pd

# Document 61개에서 page_content만 추출
chunk_texts: list[str] = [chunk.page_content for chunk in chunks]

# 로컬 임베딩 모델을 이용해서 chunk_texts의 데이터를 벡터화
local_document_vectors: list[list[float]] =(
    local_embeddings.embed_documents(chunk_texts)
)

print('chunk text count:', len(chunk_texts))
print('document vector count:', len(local_document_vectors))
print('single vector dimension:', len(local_document_vectors[0]))

# 행은 chunk 순서, 열은 embedding 차원을 나타내는 표로 바꿔 앞부분만 확인한다.
embedding_table = pd.DataFrame(local_document_vectors)
print('embedding matrix shape:', embedding_table.shape)
display(embedding_table.iloc[:3, :8])

chunk text count: 61
document vector count: 61
single vector dimension: 384
embedding matrix shape: (61, 384)


,0,1,2,3,4,5,6,7
0,-0.072716,-0.073517,-0.028007,-0.057665,0.023159,-0.045345,0.028949,-0.026064
1,-0.082109,-0.023179,-0.034830,-0.079146,-0.017336,0.008957,0.118130,0.000210
2,-0.017724,-0.136980,-0.074485,-0.077547,-0.009162,-0.068443,0.000298,0.028616


## Vector Store와 Vector DB의 역할

Vector Store는 임베딩 벡터와 원문 Document를 보관하고 query 벡터와 가까운 문서를 찾는 검색 구성 요소이다. LangChain은 구현체가 달라도 `add_documents()`, `similarity_search()`, `as_retriever()` 같은 공통 인터페이스를 제공한다.

Vector DB는 벡터 저장과 검색을 서비스로 운영하기 위한 데이터베이스이다. 영속 저장, metadata 필터, 동시 접속, 백업과 확장 같은 운영 기능을 제공한다. 모든 Vector Store가 완전한 Vector DB에 해당하지는 않는다.

- `InMemoryVectorStore`: 현재 Python 프로세스의 메모리에만 저장한다. API 흐름을 빠르게 학습할 때 적합하다.
- `Chroma`: 로컬 폴더에 저장하거나 서버·클라우드에 연결할 수 있다. 이 단원에서는 유지보수되는 `langchain-chroma` 파트너 패키지를 사용한다.
- FAISS(Facebook AI Similarity Search): 빠른 근사 최근접 검색을 제공하는 벡터 인덱스 라이브러리이다. 자체적으로 사용자 관리, 네트워크 서비스와 백업 정책을 제공하는 Vector DB에 해당하지 않는다.

관계형 데이터베이스는 `id = 10` 같은 정확한 조건 검색에 강하고, Vector Store는 표현이 다른 문장의 의미적 유사성을 찾는 데 강하다. 실제 서비스에서는 metadata 조건 필터와 벡터 검색을 함께 사용하기도 한다. [Vector Store 공식 문서](https://docs.langchain.com/oss/python/integrations/vectorstores)에서 공통 인터페이스를 확인할 수 있다.

## chunk를 InMemoryVectorStore에 인덱싱하기

인덱싱(Indexing)은 문서 chunk를 임베딩하고 벡터·원문·metadata를 검색 가능한 구조에 미리 저장하는 준비 단계이다. 질문이 들어올 때마다 전체 문서를 다시 임베딩하지 않도록 문서가 등록되거나 변경될 때 수행한다.

- 인덱싱 시점: `Document → embedding vector → Vector Store`로 변환해 저장한다.
- 검색 시점: `query → query vector → 가까운 Document`를 반환한다.

`InMemoryVectorStore`는 현재 Python 프로세스의 메모리에만 데이터를 보관하는 LangChain 구현체이다. 설치나 서버 설정 없이 인터페이스를 학습하기 좋지만 커널을 종료하면 데이터가 사라진다.

`add_documents()`는 각 Document의 `page_content`를 임베딩 모델에 전달하고 저장된 Document ID 목록을 반환한다. 생성한 `vector_store`는 다음 직접 검색과 Retriever 실습에서 사용한다.

In [18]:

# InMemoryVectorStore 생성
# - embedding=openai_embeddings
#   Document와 검색어(query)를 벡터로 변경 시켜줄 임베딩 모델을
#   openai_embeddings로 설정
vector_store = InMemoryVectorStore(embedding=openai_embeddings)

# VectorStore에 Document 저장 + document 임베딩 벡터값도 같이 저장
# -> 각 Chunk Document의 ID가 반환된다
document_ids:list[str] = vector_store.add_documents(documents=chunks)

print("chunk count: ", len(chunks))
print("indexed Document count: ", len(document_ids))
print("첫 번째 Document ID", document_ids[0])

# Document의 ID를 이용해서 수정/삭제 대상을 식별할 때 사용

chunk count:  61
indexed Document count:  61
첫 번째 Document ID d64e323d-f745-425b-a3d3-cb983c2be23c


## Vector Store를 직접 유사도 검색하기

Retriever로 감싸기 전에 Vector Store가 실제로 어떤 문서를 선택하는지 확인한다. `similarity_search_with_score(query, k)`는 질문을 내부에서 임베딩하고 가장 가까운 Document와 점수를 `(Document, score)` 튜플로 반환한다.

`k`는 상위 몇 개의 결과를 반환할지 정한다. 점수가 거리인지 유사도인지와 값의 범위는 구현체마다 다르므로 제품을 바꾸면 공식 문서에서 점수 정의를 다시 확인해야 한다. 숫자만 보지 말고 상위 Document의 본문이 질문과 관련 있는지 함께 읽는다.

In [19]:
query = 'Which domain names are reserved for documentation examples?'

direct_results: list[tuple[Document, float]] = (
    vector_store.similarity_search_with_score(
        query=query, # Vector Store에서 검색할 내용
        k=3 # 유사도가 가장 높은 상위 3개만 조회
    )
)

for rank, (document, score) in enumerate(direct_results, start=1):
    # rank는 검색 순위이고 score는 현재 Vector Store가 계산한 query와 chunk의 근접도 값이다.
    print(f'[{rank}] score: {score:.4f}')
    print('metadata:', document.metadata)
    print('page_content:', document.page_content[:500])
    print()

[1] score: 0.6025
metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web', 'start_index': 0}
page_content: IANA-managed Reserved Domains Domains Protocols Numbers About IANA-managed Reserved Domains Certain domains are set aside, and nominally registered to “IANA”, for specific policy or technical purposes. Example domains As described in RFC 2606 and RFC 6761 , a number of domains such as example.com and example.org are maintained for documentation purposes. These domains may be used as illustrative examples in documents without prior coordination with us. They are not available for registration or 

[2] score: 0.5698
metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web', 'start_index': 1365}
page_content: (RFC documents), or contractual limitations . Domains which are described as registered to IANA or ICANN on policy grounds are not available fo

## Vector Store를 Retriever로 사용하기

Retriever는 구조화되지 않은 문자열 질문을 입력받아 관련된 `list[Document]`를 반환하는 검색 인터페이스이다. 답변 문장을 생성하거나 문서를 영구 저장하는 객체가 아니라, 질문과 관련된 근거를 찾는 역할만 담당한다.

Vector Store는 Retriever를 구현할 수 있는 한 가지 저장·검색 기술이다. BM25(Best Matching 25) 같은 키워드 검색, 데이터베이스 질의, 웹 검색 서비스도 최종적으로 `list[Document]`를 반환하도록 감싸면 Retriever로 사용할 수 있다.

- Sparse Retrieval: 단어의 출현 여부와 빈도처럼 희소한 통계 표현을 사용해 제품 코드나 고유명사 같은 정확한 키워드에 강하다.
- Dense Retrieval: 임베딩 벡터를 사용해 표현이 달라도 의미가 비슷한 문서를 찾는 데 강하다.

현재 실습은 OpenAI 임베딩을 사용하는 Dense Retrieval이다. `as_retriever()`는 Vector Store의 검색 기능을 `invoke(query)` 인터페이스로 감싸며 기본 반환값에서는 점수를 제외한다.

In [20]:
# Retriever 생성
# - InMemoryVectorStore를 Retriever로 변환
# (둘 다 LangChain Component(구성 요소))
# - 반환 구조 list[tuple[Document, float]] -> list[Document]
retriever = vector_store.as_retriever(
    # query와 의미적으로 가까운 chunk를 점수 순으로 검색
    search_type='similarity',

    # 검색 후 반환되는 Document 최대 개수
    search_kwargs={'k': 3}
)

retrieved_documents: list[Document] = retriever.invoke(query)

for rank, document in enumerate(retrieved_documents, start=1):
    print(f'[{rank}] metadata:', document.metadata)
    print(document.page_content[:500])
    print()

[1] metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web', 'start_index': 0}
IANA-managed Reserved Domains Domains Protocols Numbers About IANA-managed Reserved Domains Certain domains are set aside, and nominally registered to “IANA”, for specific policy or technical purposes. Example domains As described in RFC 2606 and RFC 6761 , a number of domains such as example.com and example.org are maintained for documentation purposes. These domains may be used as illustrative examples in documents without prior coordination with us. They are not available for registration or 

[2] metadata: {'source': 'https://www.iana.org/domains/reserved', 'title': 'IANA-managed Reserved Domains', 'source_type': 'web', 'start_index': 1365}
(RFC documents), or contractual limitations . Domains which are described as registered to IANA or ICANN on policy grounds are not available for registration or transfer, with the exception of countr

## Chroma에 벡터와 Document를 영속 저장하기

Chroma는 임베딩 벡터, 원문과 metadata를 함께 관리하고 유사도 검색을 제공하는 오픈 소스 Vector DB이다. `langchain-chroma`는 Chroma 기능을 LangChain의 Vector Store 인터페이스로 사용하는 연동 패키지이다.

- Collection: 같은 목적의 벡터와 Document를 묶는 논리적인 저장 단위이다.
- Document ID: Collection 안에서 문서를 수정·삭제하거나 다시 식별할 때 사용하는 고유값이다.
- Persistence: 메모리의 데이터를 파일로 남겨 프로세스가 종료된 뒤에도 다시 읽는 방식이다.

InMemoryVectorStore의 데이터는 커널이 종료되면 사라진다. Chroma는 `persist_directory`를 지정하면 로컬 폴더에 저장하고, 같은 `collection_name`과 임베딩 모델로 다시 열 수 있다.

이 셀은 로컬 Hugging Face 임베딩으로 모든 chunk를 저장한다. 같은 ID로 다시 실행했을 때 같은 문서를 식별할 수 있도록 chunk 순번으로 ID를 만든다. [Chroma 공식 통합 문서](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)에서 로컬·서버·클라우드 연결 방식의 차이를 확인할 수 있다.

In [21]:
# ChromaDB 저장 위치 지정
chroma_directory = Path('data/chroma_retriever_lecture')

# Chroma 객체 생성
chroma_db = Chroma(
    # 같은 저장 폴더 내에서
    # 벡터와 Document(Chunk) 묶음을 구분하는 이름.
    # 이후 같은 폴더와 이름을 지정하며 해당 컬렉션을 다시 열 수 있다.
    collection_name = 'retriever_lecture',

    # Document.page_content와 검색어(query)를 숫자(벡터)로 변환하는 객체 지정
    # (local_embeddings == 허깅페이스에 받은 임베딩 모델)
    embedding_function = local_embeddings,

    # 컬렉션 데이터를 기록할 위치를 지정
    persist_directory = str(chroma_directory)
)

# Document(chunk) id 지정 (PK)
# -> chunks[0]의 id == chunk-0000
chunk_ids = [f'chunk-{index:04d}'  for index in range(len(chunks)) ]

# Chroma에 Document(Chunk) 61개 추가
# -> local_embeddings로 인해서 각 Document가 벡터(384차원)으로 변경
# -> 지정된 chunk_id + Document 1개 + Document 벡터(384개) 한 행으로 묶임
# -> 최종적으로는 ID 묶음 반환
db_ids = chroma_db.add_documents(
    documents=chunks, # Document 묶음
    ids=chunk_ids     # Document에 대응된는 ID(PK)
)

print('DB ID count:', len(db_ids))
print('first DB ID:', db_ids[0])
print('persist directory:', chroma_directory)


DB ID count: 61
first DB ID: chunk-0000
persist directory: data\chroma_retriever_lecture


### 저장된 Chroma 다시 열어 검색하기

새 `Chroma` 객체를 만들 때 이전과 같은 `collection_name`, `embedding_function`, `persist_directory`를 지정하면 저장된 컬렉션을 다시 연다. 검색 query도 문서 저장 때와 같은 임베딩 공간으로 바꿔야 하므로 같은 `local_embeddings` 객체를 전달한다.

실제 재시작을 확인하려면 커널을 재시작한 뒤 패키지·import·로컬 임베딩 준비 셀과 이 셀만 실행한다. 문서를 다시 추가하지 않아도 검색 결과가 나오면 영속 저장이 동작한 것이다.

In [22]:
# 저장된 Chroma DB 불러오기
reloaded_db = Chroma(
    collection_name = 'retriever_lecture',
    embedding_function = local_embeddings,
    persist_directory = str(chroma_directory)
)

# 검색(유사도 점수가 가장 높은 3개 조회, 점수는 반환 X)
db_results: list[Document] = reloaded_db.similarity_search(
    query=query,
    k=3
)

for rank, document in enumerate(db_results, start=1):
    print(f'[{rank}] metadata:', document.metadata)
    print(document.page_content[:500])
    print()

[1] metadata: {'title': 'IANA-managed Reserved Domains', 'start_index': 0, 'source_type': 'web', 'source': 'https://www.iana.org/domains/reserved'}
IANA-managed Reserved Domains Domains Protocols Numbers About IANA-managed Reserved Domains Certain domains are set aside, and nominally registered to “IANA”, for specific policy or technical purposes. Example domains As described in RFC 2606 and RFC 6761 , a number of domains such as example.com and example.org are maintained for documentation purposes. These domains may be used as illustrative examples in documents without prior coordination with us. They are not available for registration or 

[2] metadata: {'source_type': 'web', 'start_index': 1365, 'title': 'IANA-managed Reserved Domains', 'source': 'https://www.iana.org/domains/reserved'}
(RFC documents), or contractual limitations . Domains which are described as registered to IANA or ICANN on policy grounds are not available for registration or transfer, with the exception of countr

## 검색 문맥으로 답변하는 2-step RAG

2-step RAG는 첫 단계에서 검색을 끝낸 뒤 둘째 단계에서 검색 결과를 LLM에 전달한다. 검색과 생성이 분리되어 어떤 Document가 선택되었는지 확인한 후 모델을 호출할 수 있고, 질문 하나당 LLM 호출 횟수도 예측하기 쉽다.

이 셀은 Chroma를 Retriever로 바꾸고 검색된 Document를 하나의 `context` 문자열로 만든다. `ChatPromptTemplate → ChatOpenAI → StrOutputParser` 체인은 context와 질문을 받아 최종 문자열 답변을 반환한다.

- `context`: Retriever가 선택한 근거 본문과 출처이다.
- `question`: 사용자의 실제 질문이다.
- `StrOutputParser`: AIMessage에서 최종 텍스트만 꺼낸다.

문맥에 답이 없으면 추측하지 말라는 지시와 출처 표시 요구를 함께 전달한다. 이는 모델 자체의 지식을 삭제하는 것이 아니라 답변에 사용할 근거 범위를 제한하는 프롬프트 규칙이다.

이전 전처리와 검색 예제는 LangSmith로 전송하지 않는다. 마지막 2-step RAG 함수만 `tracing_context()` 안에서 실행해 `query → Retriever → Document → Prompt → LLM → answer` 흐름을 하나의 Trace로 기록한다.

- `project_name`: 선택한 Trace를 저장할 LangSmith 프로젝트 이름이다.
- `enabled=True`: `with` 블록 안에서만 트레이싱을 활성화한다.
- `wait_for_all_tracers()`: 백그라운드 전송이 끝나기 전에 셀 실행이 종료되지 않도록 기다린다.

Trace에는 질문, 검색된 문서 본문과 모델 입력·출력이 포함될 수 있다. 공개 수업 자료만 사용하고 개인정보, 사내 문서와 비밀값은 입력하지 않는다. 실행 후 LangSmith의 `retrieval-lecture` 프로젝트에서 `retrieval_rag_pipeline` Trace를 확인한다.

In [31]:

# Chroma DB 검색용 Retriever 생성
db_retriever = reloaded_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

#  LLM에게 전달할 메시지 틀 생성
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '검색 문맥만 근거로 한국어로 답한다. 문맥에 답이 없으면 모른다고 답한다. '
            '답변 마지막에 사용한 출처를 적는다.\n\n[검색 문맥]\n{context}',
        ),
        ('human', '{question}'),
    ]
)

# llm으로 Prompt 전달하고 응답을 AIMessage 형태로 받는 객체 생성
llm = ChatOpenAI(model=CHAT_MODEL_NAME, use_responses_api=True)

# AIMessage를 str 형태로 반환하는 parser 객체 생성
output_parser = StrOutputParser()

rag_chain = rag_prompt | llm | output_parser


# 사용자의 질문을 받아 ChromaDB에서 검색 후 반환 값을 LLM에 전달하는 함수
def run_retrieval_rag(question:str) -> dict:

    # 1) ChromaDB에 등록된 임베딩 모델이 question을 벡터화
    # 2) 벡터화된 question을 이용해서
    #    유사도가 높은 Document(Chunk) 3개를 반환
    documents: list[Document] = db_retriever.invoke(question)

    # 각 Document를 '[출처, 페이지] + 본문' 문자열로 가공

    context_parts: list[str] = [] # 가공된 문자열을 저장할 list

    # Document -> 문자열 가공
    for document in documents:
        source = document.metadata.get('source', 'unknown')
        page = document.metadata.get('page', 'web')
        context_parts.append(
            f'[출처: {source}, 페이지: {page}]\n{document.page_content}'
        )

    # 질문(question)과 검색 문맥(context_parts)을 rag_chain에 보내기
    context_text = '\n\n---\n\n'.join(context_parts)

    # Prompt 빈칸 채우기 -> LLM -> StrOutputParser -> str 반환
    answer:str = rag_chain.invoke({
        'context': context_text,
        'question': question
    })

    return {'documents': documents, 'answer':answer}

# 함수 호출
query='What strategy did Tom use on his friends when he was punished with painting the fence?'
rag_result: dict = run_retrieval_rag(query)

# 결과 나누기
rag_documents: list[Document] = rag_result['documents']
rag_answer: str = rag_result['answer']

#출력
print('retrieved source count:', len(rag_documents))
print('answer:')
print(rag_answer)

retrieved source count: 3
answer:
톰은 울타리 페인트칠을 **재미있고 특별한 일처럼 보이게 하는 전략**을 썼습니다. 친구들이 대신 칠하고 싶도록 만들어, 결국 그들과 일을 바꾸었습니다. 아이들은 울타리를 세 번 칠했고, 톰은 일을 하지 않고도 친구들과 놀 수 있었습니다.

출처: data\The_Adventures_of_Tom_Sawyer.pdf, 6·8페이지


## 검색 전략과 RAG 확장

이번에는 Dense Retrieval과 검색이 항상 생성보다 먼저 실행되는 2-step RAG의 기본 경로를 다뤘다. 검색 품질이 부족할 때는 다음 기술을 단계적으로 추가할 수 있다.

- Metadata Filter: 문서의 부서, 날짜, 유형 같은 metadata 조건으로 검색 범위를 먼저 제한한다.
- MMR(Maximal Marginal Relevance, 최대 한계 관련성): 질문 관련성뿐 아니라 결과끼리의 중복도 고려해 서로 다른 근거를 선택한다.
- Hybrid Retrieval: Dense Embedding 검색과 BM25 같은 Sparse 검색을 결합해 의미 유사성과 정확한 키워드를 함께 다룬다.
- Reranking: 첫 검색에서 넓게 가져온 후보를 별도의 모델이 다시 평가해 최종 순서를 조정한다.
- Agentic RAG: Agent가 질문과 현재 상태를 보고 Retriever를 호출할지, 어떤 검색을 추가할지 결정한다.

구조가 확장되어도 `query → list[Document] → context → answer`의 데이터 경계와 최종 답변보다 검색 문서를 먼저 검토하는 원칙은 유지된다.